<a href="https://colab.research.google.com/github/cpeters3/missionguard-ai/blob/main/Mission_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install -r requirements.txt

In [ ]:
import pandas as pd
import os


def preprocess_data(input_path="data_raw_mission_events.csv", output_path="data_processed_mission_events_clean.csv"):
    df = pd.read_csv(input_path)

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek

    df["event_length"] = df["event_message"].astype(str).apply(len)

    os.makedirs("data/processed", exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Saved {output_path}")
    return df


if __name__ == "__main__":
    preprocess_data()

In [ ]:
import pandas as pd
import random
import os
import numpy as np
from datetime import datetime, timedelta


np.random.seed(42)
random.seed(42)


def generate_event(i: int):
    base_time = datetime(2026, 1, 1) + timedelta(minutes=i)

    sensor_id = random.choice(["RADAR-1", "RADAR-2", "IR-1", "COMMS-1", "GPS-1"])
    signal_strength = np.random.normal(75, 15)
    object_speed = abs(np.random.normal(300, 120))
    object_altitude = abs(np.random.normal(10000, 5000))
    gps_drift = abs(np.random.normal(2, 1.5))
    temperature = np.random.normal(65, 10)
    communication_status = random.choice(["stable", "intermittent", "lost"])

    event_messages = {
        "normal": [
            "Routine mission event detected",
            "Stable telemetry received",
            "Nominal signal tracking"
        ],
        "suspicious_pattern": [
            "Unexpected maneuver detected",
            "Irregular signal pattern observed",
            "Rapid heading change identified"
        ],
        "communication_loss": [
            "Link degradation detected",
            "Packet drop increase observed",
            "Communication timeout event"
        ],
        "sensor_fault": [
            "Sensor calibration drift detected",
            "Telemetry inconsistency found",
            "Sensor output unstable"
        ],
        "high_priority_threat": [
            "High velocity object approaching protected zone",
            "Threat signature match detected",
            "Critical anomaly near mission area"
        ]
    }

    if signal_strength < 40 or communication_status == "lost":
        label = "communication_loss"
    elif gps_drift > 5:
        label = "sensor_fault"
    elif object_speed > 550 and object_altitude < 5000:
        label = "high_priority_threat"
    elif signal_strength < 55 or communication_status == "intermittent":
        label = "suspicious_pattern"
    else:
        label = "normal"

    event_message = random.choice(event_messages[label])

    return {
        "timestamp": base_time,
        "sensor_id": sensor_id,
        "signal_strength": round(signal_strength, 2),
        "object_speed": round(object_speed, 2),
        "object_altitude": round(object_altitude, 2),
        "gps_drift": round(gps_drift, 2),
        "temperature": round(temperature, 2),
        "communication_status": communication_status,
        "event_message": event_message,
        "label": label,
    }


def main():
    records = [generate_event(i) for i in range(2000)]
    df = pd.DataFrame(records)

    os.makedirs("data/raw", exist_ok=True)
    df.to_csv("data/raw/mission_events.csv", index=False)
    print("Saved data/raw/mission_events.csv")


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import os


def preprocess_data(input_path="data/raw/mission_events.csv", output_path="data/processed/mission_events_clean.csv"):
    df = pd.read_csv(input_path)

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek

    df["event_length"] = df["event_message"].astype(str).apply(len)

    os.makedirs("data/processed", exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Saved {output_path}")
    return df


if __name__ == "__main__":
    preprocess_data()

In [ ]:
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier


def train_model():
    df = pd.read_csv("data/processed/mission_events_clean.csv")

    target = "label"
    text_col = "event_message"
    numeric_cols = [
        "signal_strength", "object_speed", "object_altitude",
        "gps_drift", "temperature", "hour", "day_of_week", "event_length"
    ]
    categorical_cols = ["sensor_id", "communication_status"]

    X = df[[text_col] + numeric_cols + categorical_cols]
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
        ("txt", TfidfVectorizer(max_features=100), text_col)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=200, random_state=42))
    ])

    model.fit(X_train, y_train)

    os.makedirs("models", exist_ok=True)
    joblib.dump(model, "models/missionguard_model.pkl")
    print("Saved models/missionguard_model.pkl")

    return X_test, y_test, model


if __name__ == "__main__":
    train_model()

In [ ]:
import os
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from src_train import train_model


def evaluate_model():
    X_test, y_test, model = train_model()
    y_pred = model.predict(X_test)

    os.makedirs("outputs", exist_ok=True)

    report = classification_report(y_test, y_pred)
    with open("outputs/classification_report.txt", "w") as f:
        f.write(report)

    ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
    plt.tight_layout()
    plt.savefig("outputs/confusion_matrix.png")

    print(report)
    print("Saved outputs/classification_report.txt")
    print("Saved outputs/confusion_matrix.png")


if __name__ == "__main__":
    evaluate_model()

In [ ]:
import joblib
import pandas as pd


def predict_sample():
    model = joblib.load("models/missionguard_model.pkl")

    sample = pd.DataFrame([
        {
            "event_message": "High velocity object approaching protected zone",
            "signal_strength": 52.0,
            "object_speed": 620.0,
            "object_altitude": 3000.0,
            "gps_drift": 1.5,
            "temperature": 71.0,
            "hour": 14,
            "day_of_week": 2,
            "event_length": 45,
            "sensor_id": "RADAR-1",
            "communication_status": "intermittent"
        }
    ])

    prediction = model.predict(sample)[0]
    print(f"Prediction: {prediction}")


if __name__ == "__main__":
    predict_sample()

In [ ]:
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="MissionGuard AI API")
model = joblib.load("models/missionguard_model.pkl")


class MissionEvent(BaseModel):
    event_message: str
    signal_strength: float
    object_speed: float
    object_altitude: float
    gps_drift: float
    temperature: float
    hour: int
    day_of_week: int
    event_length: int
    sensor_id: str
    communication_status: str


@app.get("/")
def home():
    return {"message": "MissionGuard AI API is running"}


@app.post("/predict")
def predict(event: MissionEvent):
    df = pd.DataFrame([event.dict()])
    prediction = model.predict(df)[0]
    return {"prediction": prediction}

In [ ]:
!uvicorn src_api:app --reload

In [ ]:
from src_preprocess import preprocess_data
import pandas as pd
import os


def test_preprocess_data_creates_columns(tmp_path):
    sample = pd.DataFrame([
        {
            "timestamp": "2026-01-01 12:00:00",
            "sensor_id": "RADAR-1",
            "signal_strength": 80,
            "object_speed": 250,
            "object_altitude": 10000,
            "gps_drift": 1.2,
            "temperature": 65,
            "communication_status": "stable",
            "event_message": "Nominal signal tracking",
            "label": "normal"
        }
    ])

    input_file = tmp_path / "sample.csv"
    output_file = tmp_path / "clean.csv"
    sample.to_csv(input_file, index=False)

    df = preprocess_data(str(input_file), str(output_file))

    assert "hour" in df.columns
    assert "day_of_week" in df.columns
    assert "event_length" in df.columns
    assert os.path.exists(output_file)



In [ ]:
!pytest

In [ ]:
%%writefile tests_test_preprocess.py
from src_preprocess import preprocess_data
import pandas as pd
import os


def test_preprocess_data_creates_columns(tmp_path):
    sample = pd.DataFrame([
        {
            "timestamp": "2026-01-01 12:00:00",
            "sensor_id": "RADAR-1",
            "signal_strength": 80,
            "object_speed": 250,
            "object_altitude": 10000,
            "gps_drift": 1.2,
            "temperature": 65,
            "communication_status": "stable",
            "event_message": "Nominal signal tracking",
            "label": "normal"
        }
    ])

    input_file = tmp_path / "sample.csv"
    output_file = tmp_path / "clean.csv"
    sample.to_csv(input_file, index=False)

    df = preprocess_data(str(input_file), str(output_file))

    assert "hour" in df.columns
    assert "day_of_week" in df.columns
    assert "event_length" in df.columns
    assert os.path.exists(output_file)
